### Scenario 1

#### Querying source

In [0]:
%sql
select * from pyspark_cata.source.products

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

In [0]:
# one way: df = spark.read.table('pyspark_cata.source.products')

df = spark.sql("select * from pyspark_cata.source.products")

# Dedup
df = df.withColumn('dedup',row_number().over(Window.partitionBy('id').orderBy(desc('updatedDate'))))

df = df.filter(col('dedup')==1).drop('dedup')

display(df)

#### UPSERT

##### volume is related to pyspark, it is the destination where we can hold any kind of data, it is kid of managed cloud location

In [0]:
# Creating delta object

from delta.tables import DeltaTable
    # check if target table exists
if DeltaTable.isDeltaTable(spark,'/Volumes/pyspark_cata/source/db_volume/products_sink/'):

    dlt_obj = DeltaTable.forPath(spark,'/Volumes/pyspark_cata/source/db_volume/products_sink/')
    # even if table is empty, merge will insert all source rows
    dlt_obj.alias('tgt').merge(
        df.alias('src'),
        "src.id = tgt.id")\
        .whenMatchedUpdateAll(condition="src.updatedDate>=tgt.updatedDate")\
        .whenNotMatchedInsertAll()\
        .execute()
    print('This is upserting now')

else:
    # Target table doesn't exist - first time load
    df.write.format('delta')\
            .mode('Overwrite')\
            .save('/Volumes/pyspark_cata/source/db_volume/products_sink/')

In [0]:
%sql
select * from Delta.`/Volumes/pyspark_cata/source/db_volume/products_sink/`